In [15]:
import re
from collections import defaultdict
import math

# Paste your LaTeX table text here as a string
latex_text = """
\begin{tabular}{ c | c | c | c c c c c c}
\toprule
Dataset & Editing Method & Layer Selection & Rewrite & Rephrase & Locality & Portability & Fluency & Overall \\
\midrule

\multirow{6}{*}{\shortstack[c]{ZSRE}}
& \multirow{2}{*}{\shortstack[c]{R-ROME}}
& CMA & 0.9665 & 0.7467 & {0.9608} & {0.4837} & -- & 0.6942 \\
&      & LGA (Ours) & {0.9851} & {0.8151} & 0.9543 & {0.4757} & -- & 0.7106 \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{EMMET}
& CMA & {0.9850} & {0.8796} & {0.6589} & {0.4202} & -- & {0.7359} \\
&      & LGA (Ours) & {0.9862} & {0.8833} & {0.6857} & {0.3947} & -- & 0.7375 \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{ROME}
& CMA & {0.9669} & {0.7464} & {0.9604} & {0.4832} & -- & {0.7892} \\
&      & LGA (Ours) & {0.9862} & {0.8178} & {0.9559} & {0.4778} & -- & 0.8094 \\
\midrule

\multirow{6}{*}{\shortstack[c]{WikiBio}}
& \multirow{2}{*}{R-ROME}
& CMA & {0.6894} & -- & {0.5692} & -- & 0.8927 & 0.4553 \\
&      & LGA (Ours) & {0.7484} & -- & 0.5587 & -- & {0.8906} & 0.4722 \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{EMMET}
& CMA & {0.8258} & -- & {0.3894} & -- & {0.6321} & 0.4890 \\
&      & LGA (Ours) & {0.8433} & -- & {0.2793} & -- & {0.7069} & 0.4371 \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{ROME}
& CMA & {0.6911} & -- & {0.2902} & -- & {0.8189} & {0.3287} \\
&      & LGA (Ours) & {0.7487} & -- & {0.2851} & -- & {0.8266} & 0.3545 \\
\midrule

\multirow{6}{*}{\shortstack[c]{WikiCounterfact}}
& \multirow{2}{*}{R-ROME}
& CMA & 0.9135 & -- & 0.5601 & 0.2651 & 0.9080 & 0.4637 \\
&      & LGA (Ours) & {0.9330} & -- & {0.5669} & {0.2663} & {0.9072} & 0.4711 \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{EMMET}
& CMA & {0.9713} & -- & {0.3931} & {0.2695} & {0.8060} & {0.4630} \\
&      & LGA (Ours) & {0.9841} & -- & {0.4075} & {0.2831} & {0.8365} & 0.4651 \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{ROME}
& CMA & {0.9150} & -- & {0.5582} & {0.2635} & {0.9063} & {0.4636} \\
&      & LGA (Ours) & {0.9379} & -- & {0.5695} & {0.2659} & {0.9036} & 0.4738 \\
\midrule

\multirow{6}{*}{\shortstack[c]{WikiRecent}}
& \multirow{2}{*}{R-ROME}
& CMA & {0.9622} & -- & {0.6124} & {0.3038} & 0.9084 & 0.4925 \\
&      & LGA (Ours) & {0.9735} & -- & 0.6058 & {0.3161} & {0.9048} & 0.4977 \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{EMMET}
& CMA & {0.9837} & -- & {0.5029} & {0.2979} & {0.8612} & {0.4808} \\
&      & LGA (Ours) & {0.9892} & -- & {0.5199} & {0.3044} & {0.8534} & 0.4900 \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{ROME}
& CMA & {0.9674} & -- & {0.6105} & {0.3033} & {0.9106} & {0.4927} \\
&      & LGA (Ours) & {0.9773} & -- & {0.6027} & {0.3149} & {0.9077} & 0.4968 \\
"""

lines = [line.strip() for line in latex_text.splitlines() if line.strip()]

data = defaultdict(lambda: defaultdict(dict))
current_dataset = None
metrics = ["Rewrite", "Rephrase", "Locality", "Portability", "Fluency", "Overall"]

for line in lines:
    # Detect dataset
    dataset_match = re.search(r"\\multirow\{.*?\}\{.*?\}\{\\shortstack\[c\]\{(.+?)\}\}", line)
    if dataset_match:
        current_dataset = dataset_match.group(1)
        continue

    # Match any line that has a method and metrics
    # This regex captures lines like "& & CMA & 0.9665 & 0.7467 & ..."
    row_match = re.match(r"(?:&\s*)+([A-Za-z0-9 \(\)]+)\s*&\s*(.+)\\\\", line)
    if row_match and current_dataset:
        method = row_match.group(1).strip()
        values_str = row_match.group(2)
        values = [v.strip().replace('{','').replace('}','').replace('\\textbf','') for v in values_str.split('&')]
        values = [float(v) if v != '--' else float('nan') for v in values]
        data[current_dataset][method] = dict(zip(metrics, values))

# Compute % improvement vs CMA
improvements = defaultdict(lambda: defaultdict(dict))
for dataset, methods in data.items():
    if 'CMA' not in methods or 'LGA (Ours)' not in methods:
        continue
    cma = methods['CMA']
    lga = methods['LGA (Ours)']
    for metric in metrics:
        cma_val = cma.get(metric)
        lga_val = lga.get(metric)
        if not math.isnan(cma_val) and not math.isnan(lga_val):
            improvements[dataset]['LGA (Ours)'][metric] = (lga_val - cma_val) / cma_val * 100
        else:
            improvements[dataset]['LGA (Ours)'][metric] = float('nan')

# Print results
for dataset, methods in improvements.items():
    print(f"\nDataset: {dataset}")
    for method, metric_dict in methods.items():
        print(f"  Method: {method}")
        for metric, imp in metric_dict.items():
            print(f"    {metric}: {imp:.2f}%")

In [16]:
import re
from collections import defaultdict
import math

# latex_text = """PASTE YOUR LATEX TABLE HERE"""

metrics = ["Rewrite", "Rephrase", "Locality", "Portability", "Fluency", "Overall"]

def clean_val(v):
    """Clean LaTeX commands and convert to float"""
    v = re.sub(r'\\[a-zA-Z]+\{?', '', v)  # remove LaTeX commands like \textbf{
    v = v.replace('}', '').replace('{','').strip()
    if v == '--' or v == '':
        return float('nan')
    try:
        return float(v)
    except:
        return float('nan')

lines = [line.strip() for line in latex_text.splitlines() if line.strip()]

# dataset -> editing_method_group -> method -> metrics
data = defaultdict(lambda: defaultdict(dict))
current_dataset = None
current_editing_method = None

for line in lines:
    if line.startswith(r'\cmidrule') or line.startswith(r'\midrule') or line.startswith(r'\toprule'):
        continue

    # Detect dataset
    dataset_match = re.search(r'\\multirow\{.*?\}\{.*?\}\{\\shortstack\[c\]\{(.+?)\}\}', line)
    if dataset_match:
        current_dataset = dataset_match.group(1).strip()
        continue

    # Detect editing method group
    edit_match = re.search(r'\\multirow\{.*?\}\{.*?\}\{([A-Za-z0-9 \(\)-]+)\}', line)
    if edit_match:
        current_editing_method = edit_match.group(1).strip()
        continue

    # Detect rows with CMA or LGA
    if 'CMA' in line or 'LGA' in line:
        parts = [p.strip() for p in line.split('&') if p.strip()]
        # Method name is first part containing CMA or LGA
        method_name = next((p for p in parts if 'CMA' in p or 'LGA' in p), None)
        if method_name is None:
            continue

        # Extract all numeric or '--' values in order
        metric_values = []
        for p in parts:
            if re.match(r'^[0-9\.\-]+$', clean_val(p).__str__()) or '--' in p or re.match(r'\{?[\d\.]+\}?', p):
                metric_values.append(clean_val(p))

        # Take the last 6 numeric values for metrics
        if len(metric_values) < 6:
            # pad with nan if missing
            metric_values = [float('nan')] * (6 - len(metric_values)) + metric_values
        else:
            metric_values = metric_values[-6:]

        key = current_editing_method if current_editing_method else "Unknown"
        if key not in data[current_dataset]:
            data[current_dataset][key] = {}
        data[current_dataset][key][method_name] = dict(zip(metrics, metric_values))

# Compute % improvement of LGA vs CMA
improvements = defaultdict(lambda: defaultdict(dict))
for dataset, edit_groups in data.items():
    for group, methods in edit_groups.items():
        if 'CMA' not in methods or 'LGA (Ours)' not in methods:
            continue
        cma_vals = methods['CMA']
        lga_vals = methods['LGA (Ours)']
        improvements[dataset][group] = {}
        for metric in metrics:
            cma_val = cma_vals.get(metric)
            lga_val = lga_vals.get(metric)
            if cma_val is not None and lga_val is not None and not math.isnan(cma_val) and not math.isnan(lga_val):
                improvements[dataset][group][metric] = (lga_val - cma_val) / cma_val * 100
            else:
                improvements[dataset][group][metric] = float('nan')

# Print results
for dataset, edit_groups in improvements.items():
    print(f"\nDataset: {dataset}")
    for group, metric_dict in edit_groups.items():
        print(f"  Editing Method Group: {group}")
        for metric, imp in metric_dict.items():
            print(f"    {metric}: {imp:.2f}%")



Dataset: R-ROME
  Editing Method Group: Unknown
    Rewrite: 1.92%
    Rephrase: 9.16%
    Locality: -0.68%
    Portability: -1.65%
    Fluency: nan%
    Overall: nan%
  Editing Method Group: EMMET
    Rewrite: 0.12%
    Rephrase: 0.42%
    Locality: 4.07%
    Portability: -6.07%
    Fluency: nan%
    Overall: nan%
  Editing Method Group: ROME
    Rewrite: 2.00%
    Rephrase: 9.57%
    Locality: -0.47%
    Portability: -1.12%
    Fluency: nan%
    Overall: nan%

Dataset: WikiBio
  Editing Method Group: R-ROME
    Rewrite: 8.56%
    Rephrase: nan%
    Locality: -1.84%
    Portability: nan%
    Fluency: -0.24%
    Overall: nan%
  Editing Method Group: EMMET
    Rewrite: 2.12%
    Rephrase: nan%
    Locality: -28.27%
    Portability: nan%
    Fluency: 11.83%
    Overall: nan%
  Editing Method Group: ROME
    Rewrite: 8.33%
    Rephrase: nan%
    Locality: -1.76%
    Portability: nan%
    Fluency: 0.94%
    Overall: nan%

Dataset: WikiCounterfact
  Editing Method Group: R-ROME
    Rewrite

In [17]:
import re
import math
from collections import defaultdict

# The specific metrics we are looking for
metrics = ["Rewrite", "Rephrase", "Locality", "Portability", "Fluency", "Overall"]

def clean_val(v):
    """Removes LaTeX junk and converts to float."""
    if not v: return float('nan')
    # Remove commands like \textbf, \shortstack, \underline
    v = re.sub(r'\\[a-zA-Z]+', '', v)
    # Remove braces, slashes, and whitespace
    v = v.replace('{', '').replace('}', '').replace('\\', '').strip()
    if v in ['', '--', '-', 'None']:
        return float('nan')
    try:
        return float(v)
    except:
        return float('nan')


# --- PARSING ---
data = defaultdict(lambda: defaultdict(dict))
current_ds = "Unknown"
current_meth = "Unknown"

for line in latex_text.splitlines():
    line = line.strip()
    if not line: continue

    # Detect Dataset Name (Keywords)
    for name in ["ZSRE", "WikiBio", "WikiCounterfact", "WikiRecent"]:
        if name in line: current_ds = name
    
    # Detect Method Name (Keywords)
    for meth in ["R-ROME", "EMMET", "ROME"]:
        if f"{{{meth}}}" in line or f"shortstack[c]{{{meth}}}" in line:
            current_meth = meth

    # Extract Data Rows
    if 'CMA' in line or 'LGA (Ours)' in line:
        # Split by '&' and take the last 6 values
        cols = [c.strip() for c in line.split('&')]
        raw_vals = cols[-6:]
        numeric_vals = [clean_val(v) for v in raw_vals]
        
        row_type = "CMA" if "CMA" in line else "LGA"
        data[current_ds][current_meth][row_type] = dict(zip(metrics, numeric_vals))

# --- CALCULATION & OUTPUT ---
print(f"{'Dataset':<16} | {'Method':<8} | {'Metric':<12} | {'% Improvement'}")
print("-" * 60)

for ds, methods in data.items():
    for m_name, results in methods.items():
        if "CMA" in results and "LGA" in results:
            cma = results["CMA"]
            lga = results["LGA"]
            
            for m in metrics:
                c_val = cma[m]
                l_val = lga[m]
                
                if not math.isnan(c_val) and not math.isnan(l_val) and c_val != 0:
                    imp = ((l_val - c_val) / c_val) * 100
                    print(f"{ds:<16} | {m_name:<8} | {m:<12} | {imp:>+8.2f}%")

Dataset          | Method   | Metric       | % Improvement
------------------------------------------------------------
ZSRE             | R-ROME   | Rewrite      |    +1.92%
ZSRE             | R-ROME   | Rephrase     |    +9.16%
ZSRE             | R-ROME   | Locality     |    -0.68%
ZSRE             | R-ROME   | Portability  |    -1.65%
ZSRE             | R-ROME   | Overall      |    +2.36%
ZSRE             | EMMET    | Rewrite      |    +0.12%
ZSRE             | EMMET    | Rephrase     |    +0.42%
ZSRE             | EMMET    | Locality     |    +4.07%
ZSRE             | EMMET    | Portability  |    -6.07%
ZSRE             | EMMET    | Overall      |    +0.22%
ZSRE             | ROME     | Rewrite      |    +2.00%
ZSRE             | ROME     | Rephrase     |    +9.57%
ZSRE             | ROME     | Locality     |    -0.47%
ZSRE             | ROME     | Portability  |    -1.12%
ZSRE             | ROME     | Overall      |    +2.56%
WikiBio          | R-ROME   | Rewrite      |    +8.56%


In [19]:
import math
from collections import defaultdict
import numpy as np

metrics = ["Rewrite", "Rephrase", "Locality", "Portability", "Fluency", "Overall"]

# Step 1: Compute % improvements and store per method group
# method_group -> metric -> list of improvements across datasets
improvements_by_group = defaultdict(lambda: defaultdict(list))

for dataset, methods in data.items():
    for group, results in methods.items():
        # Only calculate if both CMA and LGA exist
        if "CMA" in results and "LGA" in results:
            cma = results["CMA"]
            lga = results["LGA"]
            for metric in metrics:
                c_val = cma.get(metric, float('nan'))
                l_val = lga.get(metric, float('nan'))
                if not math.isnan(c_val) and not math.isnan(l_val) and c_val != 0 and c_val<=l_val:
                    imp = (l_val - c_val) / c_val * 100
                    improvements_by_group[group][metric].append(imp)
                else:
                    # skip NaNs
                    continue

# Step 2: Compute mean and std for each method group & metric
print(f"{'Method':<8} | {'Metric':<12} | {'Mean % Improvement':<20} | {'Std Dev':<10}")
print("-" * 60)

for group, metric_dict in improvements_by_group.items():
    for metric, values in metric_dict.items():
        if values:
            mean_imp = np.mean(values)
            std_imp = np.std(values, ddof=0)  # population std
            print(f"{group:<8} | {metric:<12} | {mean_imp:>18.2f}% | {std_imp:>8.2f}")
        else:
            print(f"{group:<8} | {metric:<12} | {'NaN':>18} | {'NaN':>8}")



Method   | Metric       | Mean % Improvement   | Std Dev   
------------------------------------------------------------
R-ROME   | Rewrite      |               3.45% |     2.97
R-ROME   | Rephrase     |               9.16% |     0.00
R-ROME   | Overall      |               2.18% |     1.00
R-ROME   | Locality     |               1.21% |     0.00
R-ROME   | Portability  |               2.25% |     1.80
EMMET    | Rewrite      |               1.03% |     0.76
EMMET    | Rephrase     |               0.42% |     0.00
EMMET    | Locality     |               3.70% |     0.28
EMMET    | Overall      |               0.86% |     0.75
EMMET    | Fluency      |               7.81% |     4.02
EMMET    | Portability  |               3.61% |     1.43
ROME     | Rewrite      |               3.46% |     2.86
ROME     | Rephrase     |               9.57% |     0.00
ROME     | Overall      |               3.36% |     2.67
ROME     | Fluency      |               0.94% |     0.00
ROME     | Locality     

In [26]:
latex_text = """
\begin{table*}[t]
\centering
\small
%\renewcommand{\arraystretch}{0.95}
\setlength{\tabcolsep}{2.5pt}
\caption{The table shows the editing performance across "Rewrite Accuracy"m "Rephrase Accuracy", "Locality", "Portability" and "Fluency" for editing a layer selected by CMA approach and Layer Gradient Analysis (LGA) based on a small proxy set and evaluated on the test set. Here, "--" represents the dataset doesn't support the corresponding metric. We also report the overall metric which is the average of the normalized performance. We see that our proposed approach LGA consistently outperforms CMA.}
\begin{tabular}{ c | c | c | c c c c c c}
\toprule
Dataset & Model & Method & Rewrite & Rephrase & Locality & Portability & Fluency & Overall \\
\midrule

\multirow{6}{*}{\shortstack[c]{ZSRE}}
& \multirow{2}{*}{\shortstack[c]{GPT2-XL}}
& CMA & 0.9665 & 0.7467 & {0.9608} & {0.4837} & -- & 0.6942 \\
&      & LGA (Ours) & {0.9851} & {0.8151} & 0.9543 & {0.4757} & -- & \textbf{0.7106} \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{LLAMA2-7B}
& CMA & 0.9587 & 0.8859 & {0.9923} & 0.5827 & -- & 0.8549 \\
&      & LGA (Ours) & {0.9625} & {0.8988} & {0.9927} & {0.5950} & -- & \textbf{0.8623} \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{GEMMA3-12B}
& CMA & 0.8581 & 0.7041 & {0.9421} & 0.4989 & -- & 0.7508 \\
&      & LGA (Ours) & {0.9535} & {0.8796} & 0.7736 & {0.5198} & -- & \textbf{0.7816} \\
\midrule

\multirow{6}{*}{\shortstack[c]{WikiBio}}
& \multirow{2}{*}{GPT2-XL}
& CMA & {0.6894} & -- & {0.5692} & -- & 0.8927 & 0.4553 \\
&      & LGA (Ours) & {0.7484} & -- & 0.5587 & -- & {0.8906} & \textbf{0.4722} \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{LLaMA2-7B}
& CMA & 0.8818 & -- & {0.6515} & -- & {0.8547} & 0.5595 \\
&      & LGA (Ours) & {0.9560} & -- & 0.6357 & -- & 0.8597 & \textbf{0.5774} \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{GEMMA3-12B}
& CMA & 0.6817 & -- & 0.6546 & -- & 0.6683 & 0.5560 \\
&      & LGA (Ours) & {0.8124} & -- & {0.8001} & -- & {0.6666} & \textbf{0.6486} \\
\midrule

\multirow{6}{*}{\shortstack[c]{WikiCounterfact}}
& \multirow{2}{*}{GPT2-XL}
& CMA & 0.9135 & -- & 0.5601 & 0.2651 & 0.9080 & 0.4637 \\
&      & LGA (Ours) & {0.9330} & -- & {0.5669} & {0.2663} & {0.9072} & \textbf{0.4711} \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{LLaMA27B}
& CMA & {0.9902} & -- & {0.4858} & {0.5789} & 0.8824 & \textbf{0.5414} \\
&      & LGA (Ours) & {0.9911} & -- & 0.4525 & {0.5899} & {0.8788} & 0.5363 \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{GEMMA3-12B}
& CMA & 0.8276 & -- & {0.5265} & 0.4473 & {0.4871} & 0.5823 \\
&      & LGA (Ours) & {0.9290} & -- & {0.5746} & {0.4771} & {0.4871} & \textbf{0.6275} \\
\midrule

\multirow{6}{*}{\shortstack[c]{WikiRecent}}
& \multirow{2}{*}{GPT2-XL}
& CMA & {0.9622} & -- & {0.6124} & {0.3038} & 0.9084 & 0.4925 \\
&      & LGA (Ours) & {0.9735} & -- & 0.6058 & {0.3161} & {0.9048} & \textbf{0.4977} \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{LLaMA2-7B}
& CMA & {0.9748} & -- & {0.5629} & {0.5731} & {0.8718} & \textbf{0.5598} \\
&      & LGA (Ours) & {0.9750} & -- & 0.5560 & {0.5699} & 0.8771 & 0.5559 \\
\cmidrule(lr){2-9}
& \multirow{2}{*}{GEMMA3-12B}
& CMA & {0.9710} & -- & {0.5796} & {0.5216} & {0.5020} & {0.6426} \\
&      & LGA (Ours) & {0.9775} & -- & {0.5779} & {0.5327} & {0.5013} & \textbf{0.6467} \\

\bottomrule
\end{tabular}
\label{tab:cma_vs_approach_all}
\end{table*}
"""

In [ ]:
import re
from collections import defaultdict
import math
import numpy as np

metrics = ["Rewrite", "Rephrase", "Locality", "Portability", "Fluency", "Overall"]

def clean_val(v):
    """Clean LaTeX numeric value."""
    if not v: return float('nan')
    v = re.sub(r'\\[a-zA-Z]+', '', v)  # remove commands
    v = v.replace('{','').replace('}','').replace('\\','').strip()
    if v in ['', '--', '-', 'None']: return float('nan')
    try: return float(v)
    except: return float('nan')

# Nested dict: dataset -> model -> method -> metric
data = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
current_dataset = None
current_model = None

for line in latex_text.splitlines():
    line = line.strip()
    if not line or line.startswith(r'\cmidrule') or line.startswith(r'\midrule') \
       or line.startswith(r'\toprule') or line.startswith(r'\bottomrule'):
        continue

    # Detect dataset
    for ds_name in ["ZSRE", "WikiBio", "WikiCounterfact", "WikiRecent"]:
        if ds_name in line:
            current_dataset = ds_name
            break

    # Detect model
    for model_name in ["GPT2-XL", "LLAMA2-7B", "LLaMA27B", "GEMMA3-12B"]:
        if f"{{{model_name}}}" in line or f"shortstack[c]{{{model_name}}}" in line:
            current_model = model_name
            break

    # Skip if dataset or model not yet identified
    if not current_dataset or not current_model:
        continue

    # CMA or LGA row
    if 'CMA' in line or 'LGA (Ours)' in line:
        parts = [p.strip() for p in line.split('&')]
        method_name = "CMA" if "CMA" in line else "LGA"

        raw_metrics = parts[-6:]
        numeric_metrics = [clean_val(v) for v in raw_metrics]
        # Pad if some metrics missing
        if len(numeric_metrics) < len(metrics):
            numeric_metrics += [float('nan')] * (len(metrics) - len(numeric_metrics))

        # Store
        for i, metric in enumerate(metrics):
            data[current_dataset][current_model][method_name][metric] = numeric_metrics[i]

# --- VERIFY ---
for ds, models in data.items():
    print(f"\nDataset: {ds}")
    for model, methods in models.items():
        print(f"  Model: {model}")
        for method, metric_vals in methods.items():
            print(f"    Method: {method}")
            for metric, val in metric_vals.items():
                print(f"      {metric}: {val}")




Dataset: ZSRE
  Model: GPT2-XL
    Method: CMA
      Rewrite: 0.9665
      Rephrase: 0.7467
      Locality: 0.9608
      Portability: 0.4837
      Fluency: nan
      Overall: 0.6942
    Method: LGA
      Rewrite: 0.9851
      Rephrase: 0.8151
      Locality: 0.9543
      Portability: 0.4757
      Fluency: nan
      Overall: nan
  Model: LLAMA2-7B
    Method: CMA
      Rewrite: 0.9587
      Rephrase: 0.8859
      Locality: 0.9923
      Portability: 0.5827
      Fluency: nan
      Overall: 0.8549
    Method: LGA
      Rewrite: 0.9625
      Rephrase: 0.8988
      Locality: 0.9927
      Portability: 0.595
      Fluency: nan
      Overall: nan
  Model: GEMMA3-12B
    Method: CMA
      Rewrite: 0.8581
      Rephrase: 0.7041
      Locality: 0.9421
      Portability: 0.4989
      Fluency: nan
      Overall: 0.7508
    Method: LGA
      Rewrite: 0.9535
      Rephrase: 0.8796
      Locality: 0.7736
      Portability: 0.5198
      Fluency: nan
      Overall: nan

Dataset: WikiBio
  Model: GPT2-X

In [33]:
import math

# Compute improvements: (LGA - CMA)/CMA * 100
metrics = ["Rewrite", "Rephrase", "Locality", "Portability", "Fluency", "Overall"]

print("\n=== Improvements of LGA over CMA ===")
for ds, models in data.items():
    print(f"\nDataset: {ds}")
    for model, methods in models.items():
        if "CMA" not in methods or "LGA" not in methods:
            continue
        print(f"  Model: {model}")
        for metric in metrics:
            c_val = methods["CMA"].get(metric, float('nan'))
            l_val = methods["LGA"].get(metric, float('nan'))

            if math.isnan(c_val) or math.isnan(l_val) or c_val == 0:
                imp = float('nan')
            else:
                imp = (l_val - c_val) / c_val * 100

            print(f"    {metric}: {imp:+.2f}%")



=== Improvements of LGA over CMA ===

Dataset: ZSRE
  Model: GPT2-XL
    Rewrite: +1.92%
    Rephrase: +9.16%
    Locality: -0.68%
    Portability: -1.65%
    Fluency: +nan%
    Overall: +nan%
  Model: LLAMA2-7B
    Rewrite: +0.40%
    Rephrase: +1.46%
    Locality: +0.04%
    Portability: +2.11%
    Fluency: +nan%
    Overall: +nan%
  Model: GEMMA3-12B
    Rewrite: +11.12%
    Rephrase: +24.93%
    Locality: -17.89%
    Portability: +4.19%
    Fluency: +nan%
    Overall: +nan%

Dataset: WikiBio
  Model: GPT2-XL
    Rewrite: +8.41%
    Rephrase: +nan%
    Locality: -2.43%
    Portability: +nan%
    Fluency: +0.59%
    Overall: +nan%
  Model: GEMMA3-12B
    Rewrite: +19.17%
    Rephrase: +nan%
    Locality: +22.23%
    Portability: +nan%
    Fluency: -0.25%
    Overall: +nan%

Dataset: WikiCounterfact
  Model: GPT2-XL
    Rewrite: +2.13%
    Rephrase: +nan%
    Locality: +1.21%
    Portability: +0.45%
    Fluency: -0.09%
    Overall: +nan%
  Model: LLaMA27B
    Rewrite: +0.09%
    Reph

In [29]:
import re
from collections import defaultdict
import math
import statistics

# The specific metrics we are looking for
metrics = ["Rewrite", "Rephrase", "Locality", "Portability", "Fluency", "Overall"]

def clean_val(v):
    """Removes LaTeX commands, braces, and trailing line breaks."""
    if not v:
        return float('nan')
    # Remove LaTeX commands like \textbf or \shortstack
    v = re.sub(r'\\[a-zA-Z]+', '', v)
    # Remove braces, backslashes (line breaks), and whitespace
    v = v.replace('{', '').replace('}', '').replace('\\', '').strip()
    
    if v in ['', '--', '-', 'None']:
        return float('nan')
    try:
        return float(v)
    except:
        return float('nan')

def normalize_model(name):
    name = name.upper()
    if "GPT2" in name: return "GPT2-XL"
    if "LLAMA2" in name: return "LLAMA2-7B"
    if "GEMMA3" in name: return "GEMMA3-12B"
    return name

data = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
current_dataset = None
current_model = None

# --- PARSING ---
for line in latex_text.splitlines():
    line = line.strip()
    # Skip formatting lines and the header row
    if not line or any(x in line for x in [r'\cmidrule', r'\midrule', r'\toprule', r'\bottomrule', 'Rewrite & Rephrase']):
        continue

    # Identify Dataset
    for ds_name in ["ZSRE", "WikiBio", "WikiCounterfact", "WikiRecent"]:
        if ds_name in line:
            current_dataset = ds_name

    # Identify Model
    for m_name in ["GPT2-XL", "LLAMA2-7B", "LLaMA27B", "GEMMA3-12B", "LLaMA2-7B"]:
        if f"{{{m_name}}}" in line or f" {m_name} " in line:
            current_model = normalize_model(m_name)

    # CMA or LGA rows
    if ('CMA' in line or 'LGA (Ours)' in line) and current_dataset:
        parts = [p.strip() for p in line.split('&')]
        method_name = "CMA" if "CMA" in line else "LGA"
        
        # Take the last 6 columns (Rewrite through Overall)
        raw_metrics = parts[-6:]
        numeric_metrics = [clean_val(v) for v in raw_metrics]
        
        for i, metric in enumerate(metrics):
            data[current_dataset][current_model][method_name][metric] = numeric_metrics[i]

# --- AGGREGATE IMPROVEMENTS ---
model_stats = defaultdict(lambda: defaultdict(list))

for ds, models in data.items():
    for model, methods in models.items():
        if "CMA" in methods and "LGA" in methods:
            for m in metrics:
                c_val = methods["CMA"][m]
                l_val = methods["LGA"][m]
                if not math.isnan(c_val) and not math.isnan(l_val) and c_val != 0:
                    imp = ((l_val - c_val) / c_val) * 100
                    model_stats[model][m].append(imp)

# --- FINAL OUTPUT ---
print(f"{'Model':<12} | {'Metric':<12} | {'Avg Imp %':<10} | {'Std Dev'}")
print("-" * 55)

for model in ["GPT2-XL", "LLAMA2-7B", "GEMMA3-12B"]:
    print(f"--- {model} ---")
    for m in metrics:
        improvements = model_stats[model][m]
        if len(improvements) > 0:
            avg = statistics.mean(improvements)
            std = statistics.stdev(improvements) if len(improvements) > 1 else 0.0
            print(f"{'':<12} | {m:<12} | {avg:>9.2f}% | {std:>7.2f}")
        else:
            print(f"{'':<12} | {m:<12} | {'N/A':>10} | {'N/A':>7}")

Model        | Metric       | Avg Imp %  | Std Dev
-------------------------------------------------------
--- GPT2-XL ---
             | Rewrite      |      3.45% |    3.43
             | Rephrase     |      9.16% |    0.00
             | Locality     |     -0.60% |    1.30
             | Portability  |      0.95% |    2.88
             | Fluency      |     -0.24% |    0.15
             | Overall      |        N/A |     N/A
--- LLAMA2-7B ---
             | Rewrite      |      2.23% |    4.13
             | Rephrase     |      1.46% |    0.00
             | Locality     |     -2.62% |    3.00
             | Portability  |      1.15% |    1.48
             | Fluency      |      0.26% |    0.58
             | Overall      |        N/A |     N/A
--- GEMMA3-12B ---
             | Rewrite      |     10.80% |    7.64
             | Rephrase     |     24.93% |    0.00
             | Locality     |      3.30% |   16.87
             | Portability  |      4.33% |    2.27
             | Fluency  